In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.silver.gho_clean")

In [0]:
from pyspark.sql import Window
w = Window.partitionBy("country_code", "indicator_code").orderBy(F.desc("year"))
gold_latest = (
    silver
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)

In [0]:
gold_wide = (
    silver
    .groupBy("country_code", "region", "year")
    .pivot("indicator_code")
    .agg(F.first("numeric_value"))
)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
gold_latest.write.format("delta")
.mode("overwrite").saveAsTable("workspace.gold.gho_latest").option("over")
gold_wide.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.gho_wide")

In [0]:
latest = spark.table("workspace.gold.gho_latest")
wide = spark.table("workspace.gold.gho_wide")

n_dup_latest = (latest.groupBy("country_code", "indicator").count()
    .filter("count > 1").count())
assert n_dup_latest == 0, f"{n_dup_latest} duplicated keys in gho_latest"

n_dup_wide = (wide.groupBy("country_code", "year").count()
    .filter("count > 1").count())
assert n_dup_wide == 0, f"{n_dup_wide} duplicated keys in gho_wide"

assert len(wide.columns) == 3 + 15, f"{len(wide.columns)} columns in gho_wide, expected 18"